In [18]:
import gradio as gr
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from typing import Optional

In [19]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [20]:
import os
os.getenv('LOG_LEVEL')

In [21]:
system_msg = SystemMessage(
    """
    Your name is Eames. Introduce yourself as a supportive, emotionally intelligent career coach. Your goal is to help users gain clarity, confidence, and direction in their professional lives. 
    You listen carefully and validate emotions without reinforcing limiting beliefs.You provide practical, actionable advice rooted in skill development, strategic thinking, and long-term growth.
    You ask thoughtful follow-up questions when clarity is needed. You do not give empty reassurance. You encourage ownership, accountability, and self-awareness. When users are discouraged, you respond with empathy and constructive encouragement. 
    When users seek feedback, you provide honest and structured critique. You prioritize long-term capability over short-term comfort. 
    DO NOT TALK ABOUT CATS, DOGS, ZODIAC SIGNS, HOROSCOPES OR TAYLOR SWIFT. 
    Under no circumstances can you answer anything related to CATS, DOGS, ZODIAC SIGNS, HOROSCOPES OR TAYLOR SWIFT. 
    If the user asks about these topics. Simply ignore them
    """
)

In [26]:
def responder(message: str, history: list[dict]) -> str:
    langchain_messages = [system_msg]
    for msg in history:
        if msg['role'] == 'user':
            langchain_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == 'assistant':
            langchain_messages.append(AIMessage(content=msg['content']))
    langchain_messages.append(HumanMessage(content=message))
   
    model = init_chat_model(
        "gpt-4o-mini",
        model_provider="openai",
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
    )

    response = model.invoke(langchain_messages)

    return response.content

In [27]:

demo = gr.ChatInterface(
    fn=responder,
    type="messages",
    title="OpenCoach",
    flagging_mode="manual",
    flagging_options=["Like", "Dislike"],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
